# SAGE — Inference pretrained Student FMB10, bản V4 chuẩn bị bàn giao 08/11/2026

Chỉnh từ notebook v3 đã sử dụng thực tế: **mỗi lần chạy có run ID riêng**, không xoá đầu ra cũ; lưu manifest và run config. **Đây là notebook sẵn sàng chạy trên Kaggle nhưng chưa được chạy trong phiên này; các số hiệu năng thật vẫn lấy từ ZIP v3 ngày 02/11.**

## Cell 1 — Khai báo cấu hình và ghi nhận môi trường

**Vì sao?** Bài nộp cần chỉ rõ GPU, phiên bản phần mềm, và chạy trên môi trường nào. Dừng sớm nếu Kaggle đang bật CPU.

In [ ]:
from pathlib import Path
import os, sys, json, platform, subprocess, shutil, time, random, hashlib
from datetime import datetime, timezone
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import torchvision
import matplotlib
import matplotlib.pyplot as plt

N_IMAGES = 10
SEED = 2
REPO_URL = 'https://github.com/RollingPlain/SAGE_IVIF.git'
UPSTREAM_COMMIT = '377fdbef6b8b6ccbd1139358ad66bb41ea084dc1'
CHECKPOINT_SHA256 = 'ecfde477ca4cd65401f207d5dc91ce6c475a49891d768dbdb7a66366255ee836'
REPO_DIR = Path('/kaggle/working/SAGE_IVIF')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f') + '_sage_fmb10_student'
OUTPUT_DIR = Path('/kaggle/working/sage_locbe_runs') / RUN_ID
FUSED_DIR = OUTPUT_DIR / 'fused'

# Nếu đã Add Input FMB trên Kaggle và muốn chọn chính xác, điền
# thư mục CHA chứa 2 thư mục con (Ir + Vis hoặc IR + Visible), ví dụ:
# DATA_ROOT = '/kaggle/input/my-fmb/FMB/test'
DATA_ROOT = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=False)  # refuse overwrite
FUSED_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('UTC:', datetime.now(timezone.utc).isoformat())
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('NumPy:', np.__version__)
print('OpenCV:', cv2.__version__)
print('Pillow:', metadata.version('Pillow'))
print('Pandas:', pd.__version__)
print('CUDA runtime (PyTorch):', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU total memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError('Chưa bật GPU: Kaggle Notebook → Settings → Accelerator → GPU rồi Run All lại.')

## Cell 2 — Lấy code và checkpoint của tác giả

**Vì sao?** Sử dụng đúng mã nguồn gốc và checkpoint tác giả cung cấp, không tải SAM (chỉ dùng trong train Teacher). Ghi commit SHA để những người khác có thể kiểm tra phiên bản.

In [ ]:
if not (REPO_DIR / '.git').is_dir():
    command = ['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)]
    print(' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Dùng lại source đã clone:', REPO_DIR)

dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
assert not dirty, 'Upstream checkout has local changes; use a clean clone.'
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', UPSTREAM_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)

repo_commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
assert repo_commit == UPSTREAM_COMMIT
checkpoint = REPO_DIR / 'checkpoints' / 'checkpoints.pt'
model_file = REPO_DIR / 'model_sub' / 'model.py'
dataset_file = REPO_DIR / 'dataset' / 'dataset_test.py'
assert checkpoint.is_file(), f'Thiếu checkpoint: {checkpoint}'
assert model_file.is_file(), f'Thiếu model Student: {model_file}'
assert dataset_file.is_file(), f'Thiếu code đọc input: {dataset_file}'

with checkpoint.open('rb') as f:
    checkpoint_sha = hashlib.file_digest(f, 'sha256').hexdigest()
assert checkpoint_sha == CHECKPOINT_SHA256, 'Checkpoint differs from measured run'
print('Commit:', repo_commit)
print('Checkpoint:', checkpoint)
print('Checkpoint SHA256:', checkpoint_sha)
print('Checkpoint size (MB):', round(checkpoint.stat().st_size / 1024**2, 3))

## Cell 2B (v3) — Tạo 10 cặp IR–Visible từ FMB, có kiểm tra tên trong TAR

- **Sửa lỗi v2:** không còn chỉ chấp nhận `.rgb.jpg` và `.ir.jpg`; nhận cả `.rgb.png`, `.ir.png`, `.jpeg` và cách đóng gói dưới thư mục khác.
- **Không tải lại TAR nếu đã có trong `/kaggle/working/sage_hf_cache`.** In 12 tên file đầu tiên và số cặp nhận diện được, để dễ tìm lỗi thực tế.
- Chỉ chuyển ảnh IR thực sang grayscale và Visible thực sang RGB rồi lưu PNG cùng tên. Không tạo IR giả, không resize nếu kích thước khác nhau.
- `DATA_ROOT = None` để tự chọn nguồn demo FMB từ `LC-State/RGB2IR_Translation` (bản đóng gói lại, không phải bằng chứng split benchmark chính thức).
- Nếu bạn đã gắn FMB chuẩn bằng Add Input, đặt `DATA_ROOT` thành thư mục cha `Ir/Vis` ở Cell 1 để không tự tải.


In [ ]:
# Cell 2B v3 — Tái sử dụng TAR đã tải, dò cấu trúc WebDataset và lấy đúng 10 cặp FMB.
from pathlib import Path, PurePosixPath
from collections import Counter
import io, re, tarfile, json, subprocess, sys
from PIL import Image

if DATA_ROOT is None:
    try:
        from huggingface_hub import list_repo_files, hf_hub_download
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
        from huggingface_hub import list_repo_files, hf_hub_download

    HF_REPO = 'LC-State/RGB2IR_Translation'
    DEMO_ROOT = Path('/kaggle/working/FMB10_SAGE_Demo')
    IR_DIR = DEMO_ROOT / 'Ir'
    VIS_DIR = DEMO_ROOT / 'Vis'
    HF_CACHE = Path('/kaggle/working/sage_hf_cache')
    IR_DIR.mkdir(parents=True, exist_ok=True)
    VIS_DIR.mkdir(parents=True, exist_ok=True)
    HF_CACHE.mkdir(parents=True, exist_ok=True)

    # Không dùng lại file thiếu một phía hoặc sai kích thước.
    def prepared_pairs():
        verified = set()
        for ir_file in IR_DIR.glob('*.png'):
            vis_file = VIS_DIR / ir_file.name
            if not vis_file.is_file():
                continue
            try:
                with Image.open(ir_file) as im1, Image.open(vis_file) as im2:
                    if im1.size == im2.size and im1.size[0] > 0 and im1.size[1] > 0:
                        im1.verify()
                        im2.verify()
                        verified.add(ir_file.stem)
            except Exception:
                pass
        return verified

    # Nhận diện kiểu: sample.rgb.jpg / sample.ir.png; sample/rgb.jpg;
    # rgb/sample.jpg / ir/sample.jpg; sample.rgb (không đuôi).
    MODALITIES = {
        'rgb': 'vis', 'vis': 'vis', 'visible': 'vis', 'visual': 'vis',
        'ir': 'ir', 'infrared': 'ir', 'thermal': 'ir', 'lwir': 'ir'
    }
    EXTENSIONS = {'jpg', 'jpeg', 'png', 'bmp', 'tif', 'tiff', 'webp', 'ppm', 'pgm'}

    def modality_and_key(raw_name):
        path = PurePosixPath(raw_name)
        components = path.name.split('.')
        if components and components[-1].lower() in EXTENSIONS:
            components.pop()
        if not components:
            return None
        # WebDataset: <sample-id>.rgb.jpg, <sample-id>.ir.png
        if len(components) >= 2 and components[-1].lower() in MODALITIES:
            key = '.'.join(components[:-1])
            return (key, MODALITIES[components[-1].lower()]) if key else None
        # Thư mục trực tiếp: rgb/<sample-id>.jpg, ir/<sample-id>.png
        parent = path.parent.name.lower()
        if parent in MODALITIES:
            key = '.'.join(components)
            return (key, MODALITIES[parent]) if key else None
        # Một số TAR đóng gói: <sample-id>/rgb.jpg, <sample-id>/ir.jpg
        if len(components) == 1 and components[0].lower() in MODALITIES:
            key = path.parent.name
            return (key, MODALITIES[components[0].lower()]) if key else None
        return None

    already = prepared_pairs()
    print(f'Cặp FMB hợp lệ đã chuẩn bị trước đó: {len(already)}/{N_IMAGES}')
    saved = []
    if len(already) < N_IMAGES:
        # Nếu phiên Kaggle trước đã tải TAR về /kaggle/working, dùng lại.
        cached = sorted(HF_CACHE.rglob('fmb-test-*.tar'))
        if cached:
            print('Đã thấy TAR trong cache:', [str(x) for x in cached])
            tar_list = [(p.name, p) for p in cached]
        else:
            repo_files = list_repo_files(HF_REPO, repo_type='dataset')
            shard_names = sorted(f for f in repo_files
                                 if f.lower().startswith('data/fmb/test/') and f.lower().endswith('.tar'))
            if not shard_names:
                raise RuntimeError('Không thấy shard TAR FMB/test của repository đã chọn.')
            print('Số shard FMB/test:', len(shard_names))
            tar_list = []
            for shard_name in shard_names:
                p = Path(hf_hub_download(repo_id=HF_REPO, repo_type='dataset',
                                         filename=shard_name, local_dir=str(HF_CACHE)))
                tar_list.append((shard_name, p))
                # Dataset FMB/test ở nguồn này chỉ có 1 shard; không tải các tập khác.

        total_recognized = 0
        total_candidates = 0
        total_bad_size = 0
        file_examples = []
        for shard_name, shard_path in tar_list:
            with tarfile.open(shard_path, 'r:*') as tar:
                members = [m for m in tar.getmembers() if m.isfile()]
                if not file_examples:
                    file_examples = [m.name for m in members[:30]]
                print(f'\nĐọc TAR: {shard_path.name} ({len(members)} files)')
                print('Ví dụ tên file đầu tiên:')
                for example in [m.name for m in members[:12]]:
                    print('  ', example)

                grouped = {}
                for member in members:
                    parsed = modality_and_key(member.name)
                    if parsed is None:
                        continue
                    key, modality = parsed
                    total_recognized += 1
                    # Nếu một key có nhiều file cùng một modality, không dùng ngẫu nhiên.
                    group = grouped.setdefault(key, {})
                    if modality in group:
                        group['ambiguous'] = True
                    group[modality] = member
                complete = [(key, entry) for key, entry in grouped.items()
                            if 'ir' in entry and 'vis' in entry and not entry.get('ambiguous')]
                total_candidates += len(complete)
                print(f'File nhận diện modality: {total_recognized}; cặp RGB-IR trong TAR: {len(complete)}')

                for key, entries in sorted(complete):
                    if len(already) >= N_IMAGES:
                        break
                    # Chỉ chấp nhận ID có ký tự rõ ràng, tránh trùng tên file do đổi dấu.
                    out_stem = re.sub(r'[^a-zA-Z0-9_-]+', '_', key).strip('_')
                    if not out_stem or out_stem in already:
                        continue
                    ir_stream = tar.extractfile(entries['ir'])
                    vis_stream = tar.extractfile(entries['vis'])
                    if ir_stream is None or vis_stream is None:
                        continue
                    try:
                        with Image.open(io.BytesIO(ir_stream.read())) as ir_raw:
                            ir_im = ir_raw.convert('L')
                        with Image.open(io.BytesIO(vis_stream.read())) as vis_raw:
                            vis_im = vis_raw.convert('RGB')
                        if ir_im.size != vis_im.size:
                            total_bad_size += 1
                            continue
                        if min(ir_im.size) < 4:
                            continue
                        target_ir = IR_DIR / (out_stem + '.png')
                        target_vis = VIS_DIR / (out_stem + '.png')
                        ir_im.save(target_ir)
                        vis_im.save(target_vis)
                        already.add(out_stem)
                        saved.append({'stem': out_stem, 'source_key': key,
                                      'source_shard': shard_name,
                                      'width': vis_im.width, 'height': vis_im.height})
                    except Exception as exc:
                        print('Không đọc được mẫu:', key, repr(exc)[:140])
            if len(already) >= N_IMAGES:
                break

        print(f'Đã nhận diện {total_recognized} file modality và {total_candidates} cặp trong TAR.')
        print(f'Số cặp bị sai kích thước: {total_bad_size}')
        if len(already) < N_IMAGES:
            raise RuntimeError(
                f'Chỉ chuẩn bị được {len(already)}/{N_IMAGES} cặp FMB. '
                'Hãy gửi 12 tên file TAR đã in ở trên để mình sửa chính xác định dạng. '
                f'Ví dụ: {file_examples[:6]}'
            )

    (DEMO_ROOT / 'source_manifest.json').write_text(json.dumps({
        'origin_repository': HF_REPO,
        'subset': 'FMB',
        'repackaged_split': 'test',
        'note': 'Split do bên thứ ba đóng gói, không khẳng định tương đương test chính thức.',
        'new_samples': saved,
        'prepared_valid_stems': sorted(already)
    }, indent=2, ensure_ascii=False), encoding='utf-8')
    DATA_ROOT = str(DEMO_ROOT)
else:
    print('Dùng nguồn ảnh người dùng gắn trên Kaggle:', DATA_ROOT)

print('DATA_ROOT =', DATA_ROOT)
if Path(DATA_ROOT, 'Ir').is_dir() and Path(DATA_ROOT, 'Vis').is_dir():
    n_ir = len(list(Path(DATA_ROOT, 'Ir').glob('*.png')))
    n_vis = len(list(Path(DATA_ROOT, 'Vis').glob('*.png')))
    print(f'Số PNG đầu vào IR: {n_ir}; Visible: {n_vis}')


## Cell 3 — Tự tìm cặp ảnh IR–VIS theo cùng tên

**Vì sao?** Fusion chỉ hợp lệ nếu hai ảnh thuộc cùng cảnh và đã căn chỉnh. Mã sẽ tìm những thư mục có cặp IR / VIS trên `/kaggle/input` hoặc trong bộ ảnh mẫu của repo. Chỉ nhận ảnh có tên gốc khớp và kích thước bằng nhau; không tự resize để che giấu lỗi dữ liệu.

**Ưu tiên:** dữ liệu FMB đã gắn từ Kaggle (nếu có), sau đó ảnh ví dụ của tác giả. Có thể đặt `DATA_ROOT` ở Cell 1 nếu muốn cố định nguồn.

In [ ]:
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
IR_NAMES = {'ir', 'infrared', 'thermal'}
VIS_NAMES = {'vis', 'vi', 'visible', 'rgb', 'visual'}

def image_files(folder):
    return sorted(p for p in Path(folder).iterdir()
                  if p.is_file() and p.suffix.lower() in VALID_EXTS)

def detect_dirs(root):
    root = Path(root)
    if not root.is_dir():
        return None
    children = [p for p in root.iterdir() if p.is_dir()]
    irs = [p for p in children if p.name.lower() in IR_NAMES]
    viss = [p for p in children if p.name.lower() in VIS_NAMES]
    return (irs[0], viss[0]) if irs and viss else None

def pair_images(root):
    dirs = detect_dirs(root)
    if not dirs:
        return [], {'no_pair_folders': True}
    ir_dir, vis_dir = dirs
    ir_files, vis_files = image_files(ir_dir), image_files(vis_dir)

    def make_unique(files):
        by_stem, dup = {}, set()
        for f in files:
            if f.stem in by_stem:
                dup.add(f.stem)
            else:
                by_stem[f.stem] = f
        for s in dup:
            by_stem.pop(s, None)
        return by_stem, dup

    ir_map, ir_dup = make_unique(ir_files)
    vis_map, vis_dup = make_unique(vis_files)
    common = sorted(set(ir_map) & set(vis_map))
    pairs, invalid = [], []
    for stem in common:
        if len(pairs) >= N_IMAGES:
            break  # Chỉ cần xác thực đủ 10 cặp, tránh đọc toàn bộ dataset lớn.
        ir, vis = ir_map[stem], vis_map[stem]
        a = cv2.imread(str(ir), cv2.IMREAD_COLOR)
        b = cv2.imread(str(vis), cv2.IMREAD_COLOR)
        if a is None or b is None or a.shape[:2] != b.shape[:2]:
            invalid.append(stem)
        else:
            pairs.append({'stem': stem, 'ir_path': str(ir), 'vis_path': str(vis),
                          'height': int(b.shape[0]), 'width': int(b.shape[1])})
    issues = {'ir_total':len(ir_files), 'vis_total':len(vis_files),
              'duplicates':sorted(ir_dup | vis_dup), 'unreadable_or_size_mismatch':invalid,
              'matching_stems':len(common), 'checked_stems':len(pairs)+len(invalid),
              'paired_valid_among_checked':len(pairs)}
    return pairs, issues

roots = []
if DATA_ROOT:
    roots = [Path(DATA_ROOT)]
else:
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for current, dirs, files in os.walk(input_root):
            relative_depth = len(Path(current).relative_to(input_root).parts)
            if relative_depth > 5:
                dirs[:] = []
                continue
            if detect_dirs(current):
                roots.append(Path(current))
                dirs[:] = []
    roots.append(REPO_DIR / 'data' / 'test')

candidates = []
for root in dict.fromkeys(roots):
    pairs, issues = pair_images(root)
    candidates.append((root, pairs, issues))
    print(f'Nguồn: {root} | cặp hợp lệ: {len(pairs)} | thông tin: {issues}')

# Chỉ chấp nhận bộ ảnh có ít nhất 10 cặp; không tái sử dụng một cặp để đủ số lượng.
choices = [(root, pairs, issues) for root, pairs, issues in candidates if len(pairs) >= N_IMAGES]
if not choices:
    raise RuntimeError(
        f'Không tìm được {N_IMAGES} cặp IR–VIS hợp lệ. Hãy vào Kaggle → Add Input '
        'gắn FMB do bạn Mân cung cấp, rồi đặt DATA_ROOT thành thư mục cha chứa Ir/Vis và Run All lại.'
    )
# Nếu có FMB, ưu tiên FMB; kế đó các bộ gắn ở /kaggle/input; cuối cùng ảnh ví dụ của repo.
choices.sort(key=lambda x: (0 if 'fmb' in str(x[0]).lower() else 1,
                            0 if str(x[0]).startswith('/kaggle/input') else 1,
                            str(x[0])))
chosen_root, all_pairs, source_issues = choices[0]
selected_pairs = all_pairs[:N_IMAGES]
print('\nCHỌN:', chosen_root)
print('Đúng 10 ảnh (10 tên khác nhau):', [p['stem'] for p in selected_pairs])
print('Lưu ý: tên trùng nhau không đủ chứng minh đã đăng ký không gian IR–VIS; phải dùng dataset đã căn chỉnh.')

## Cell 4 — Tải pretrained Student và kiểm tra checkpoint

**Vì sao?** Kiểm tra checkpoint thật sự khớp kiến trúc; không dùng `strict=False` để che thiếu tham số. `Network` nhận **Y của Visible** và **IR 1 kênh**, trả về kênh sáng hợp nhất và đặc trưng trung gian.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
from model_sub.model import Network
from dataset.dataset_test import Data
import utils

device = torch.device('cuda:0')
model = Network()
state = torch.load(checkpoint, map_location='cpu', weights_only=True)
assert isinstance(state, dict), f'Unexpected checkpoint type: {type(state)}'
model.load_state_dict(state, strict=True)
model.to(device).eval()
parameter_count = sum(p.numel() for p in model.parameters())
print('Đã load weights thành công (strict=True).')
print('Student trainable params:', parameter_count, f'({parameter_count / 1e6:.4f} M)')
print('Input: Y visible + IR | Output: fused Y -> ghép hai kênh màu Visible')

## Cell 5 — Chạy inference, đo thời gian và lưu chính xác 10 ảnh PNG

**Vì sao?** Mỗi cặp ảnh được đưa qua Student ở chế độ `eval()`/`inference_mode()` để không cập nhật trọng số. Dùng `torch.cuda.synchronize()` trước/sau phép đo để đo đúng thời gian GPU (CUDA hoạt động bất đồng bộ). `forward_ms` chỉ đo mô hình, `fusion_ms` gồm mô hình và chuyển màu, `total_ms` thêm lưu ảnh.

**Lưu ý:** Giữ đúng bước đổi màu trong `utils.YCrCb2RGB` của tác giả. Các biến `cb`/`cr` trong `dataset_test.py` đặt tên dễ gây nhầm, nhưng dữ liệu được lấy từ OpenCV `COLOR_BGR2YCrCb` và được sử dụng theo đúng thứ tự của source.

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Run ID riêng; không xóa fused PNG của run trước.
assert not list(FUSED_DIR.glob('*.png')), 'Run ID không mới hoặc output đã tồn tại'

# Dataset gốc ghép theo tên stem nhưng thứ tự không cố định (set).
# Cố định danh sách 10 stem và khớp với audit ở Cell 3 để tái lập thử nghiệm.
test_data = Data(mode='test', img_dir=str(chosen_root))
test_data.img_list = [p['stem'] for p in selected_pairs]
test_loader = DataLoader(test_data, batch_size=1, shuffle=False,
                         pin_memory=True, num_workers=0, drop_last=False)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(device)
torch.backends.cudnn.benchmark = True  # Bám sát test.py tác giả.

# Warm-up: tránh đo thời gian khởi tạo CUDA trong lần đầu.
with torch.inference_mode():
    warm = test_data[0]
    warm_y = warm['y'].unsqueeze(0).to(device)
    warm_ir = warm['ir'].unsqueeze(0).to(device)
    for _ in range(2):
        model(warm_y, warm_ir)
    torch.cuda.synchronize(device)

records = []
run_start = time.perf_counter()
with torch.inference_mode():
    for i, batch in enumerate(tqdm(test_loader, desc='SAGE inference', total=N_IMAGES)):
        item = selected_pairs[i]
        assert batch['name'][0] == item['stem'], 'Thứ tự DataLoader thay đổi ngoài dự kiến'
        t0 = time.perf_counter()
        y = batch['y'].to(device, non_blocking=True)
        ir = batch['ir'].to(device, non_blocking=True)
        cb = batch['cb'].to(device, non_blocking=True)
        cr = batch['cr'].to(device, non_blocking=True)
        torch.cuda.synchronize(device)
        start_forward = time.perf_counter()
        fused_y, _ = model(y, ir)
        torch.cuda.synchronize(device)
        end_forward = time.perf_counter()
        fused_rgb = utils.YCrCb2RGB(torch.cat((fused_y, cb, cr), dim=1)).clamp(0, 1)
        torch.cuda.synchronize(device)
        end_color = time.perf_counter()

        output_path = FUSED_DIR / (item['stem'] + '.png')
        torchvision.utils.save_image(fused_rgb, str(output_path))
        end_total = time.perf_counter()
        assert output_path.is_file(), f'Chưa lưu được: {output_path}'
        with Image.open(output_path) as im:
            assert im.format == 'PNG' and im.mode == 'RGB', (im.format, im.mode)
            assert im.size == (item['width'], item['height']), (im.size, item)

        records.append({**item,
                        'output_png': str(output_path),
                        'forward_ms': round((end_forward - start_forward) * 1000, 3),
                        'fusion_ms': round((end_color - start_forward) * 1000, 3),
                        'total_ms': round((end_total - t0) * 1000, 3)})
run_total_sec = time.perf_counter() - run_start

timing_df = pd.DataFrame(records)
timing_df.to_csv(OUTPUT_DIR / 'inference_times.csv', index=False)
actual_pngs = sorted(FUSED_DIR.glob('*.png'))
assert len(records) == N_IMAGES and len(actual_pngs) == N_IMAGES, 'Số lượng output khác 10!'
print('THÀNH CÔNG:', len(actual_pngs), 'fused PNG ở', FUSED_DIR)
print('Thời gian tổng loop:', round(run_total_sec, 3), 's')
print('Forward mean / median / p95 (ms):',
      round(timing_df.forward_ms.mean(), 3),
      round(timing_df.forward_ms.median(), 3),
      round(float(np.percentile(timing_df.forward_ms, 95)), 3))
print('GPU peak allocated (MB):', round(torch.cuda.max_memory_allocated(device) / 1024**2, 2))
display(timing_df[['stem', 'width', 'height', 'forward_ms', 'fusion_ms', 'total_ms']])

## Cell 6 — Minh họa IR / Visible / Fused

**Vì sao?** Kiểm tra bằng mắt xem ảnh fused có màu, chi tiết và vật thể có xuất hiện đúng hay không. Hình này là **minh họa định tính**, chưa phải bằng chứng mô hình đạt mIoU hoặc metric báo cáo trong paper.

In [ ]:
show_n = min(4, len(selected_pairs))
fig, axes = plt.subplots(show_n, 3, figsize=(13, show_n * 3.2), squeeze=False)
for r, item in enumerate(selected_pairs[:show_n]):
    ir_img = Image.open(item['ir_path']).convert('L')
    vis_img = Image.open(item['vis_path']).convert('RGB')
    fused_img = Image.open(FUSED_DIR / (item['stem'] + '.png')).convert('RGB')
    for j, img in enumerate([ir_img, vis_img, fused_img]):
        axes[r, j].imshow(img, cmap='gray' if j == 0 else None)
        axes[r, j].axis('off')
        axes[r, j].set_title(f"{item['stem']} — {['IR', 'Visible', 'Fused RGB'][j]}")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'preview_ir_vis_fused.png', dpi=140, bbox_inches='tight')
plt.show()

## Cell 7 — Xuất log môi trường, thống kê và nén kết quả

**Vì sao?** Ngoài notebook, nhóm trưởng cần minh chứng có thể đọc lại: tên GPU, mã nguồn, checksum checkpoint, cặp ảnh đã dùng, các thời gian từng ảnh và ảnh PNG thực tế. Kết quả được nén trong một ZIP để gửi thuận tiện.

`forward_ms` đo Student forward theo từng ảnh sau warm-up, không đại diện tốc độ tác giả báo cáo trên phần cứng khác.

In [ ]:
try:
    gpu_driver = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
        text=True, stderr=subprocess.STDOUT).strip()
except Exception as exc:
    gpu_driver = f'nvidia-smi unavailable: {exc}'

env = {
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'repository': REPO_URL,
    'repo_commit': repo_commit,
    'checkpoint_sha256': checkpoint_sha,
    'checkpoint_path': str(checkpoint),
    'dataset_root': str(chosen_root),
    'dataset_source': ('LC-State/RGB2IR_Translation FMB/test repackaged' if 'FMB10_SAGE_Demo' in str(chosen_root) else 'User-provided dataset / repository samples'),
    'dataset_audit': source_issues,
    'n_images': len(records),
    'seed': SEED,
    'python': sys.version,
    'platform': platform.platform(),
    'gpu_name': torch.cuda.get_device_name(0),
    'gpu_driver': gpu_driver,
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'torch_cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'pillow': metadata.version('Pillow'),
    'pandas': pd.__version__,
    'student_parameters': parameter_count,
    'peak_gpu_allocated_mb': round(torch.cuda.max_memory_allocated(device) / 1024**2, 3),
    'loop_total_s': round(run_total_sec, 3),
    'mean_forward_ms': round(float(timing_df.forward_ms.mean()), 3),
    'median_forward_ms': round(float(timing_df.forward_ms.median()), 3),
    'p95_forward_ms': round(float(np.percentile(timing_df.forward_ms, 95)), 3),
    'mean_fusion_ms': round(float(timing_df.fusion_ms.mean()), 3),
    'meaning_of_time': 'Warm-up 2 iterations; CUDA sync around model forward; batch=1; excludes dataset loading from forward_ms.'
}
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(env, indent=2, ensure_ascii=False), encoding='utf-8')
with (OUTPUT_DIR / 'pip_freeze.txt').open('w', encoding='utf-8') as out_f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                   stdout=out_f, stderr=subprocess.DEVNULL, check=True)

report = f"""# SAGE pretrained Student — Báo cáo chạy thử (Lộc bé)

- **Nguồn mã:** {REPO_URL}
- **Git commit:** `{repo_commit}`
- **Checkpoint SHA256:** `{checkpoint_sha}`
- **Dataset:** `{chosen_root}`\n- **Dataset provenance:** {env['dataset_source']}
- **GPU:** {env['gpu_name']}
- **PyTorch / CUDA:** {env['torch']} / {env['torch_cuda']}
- **Kết quả:** {len(records)}/10 PNG fused RGB, kích thước từng ảnh giữ nguyên.
- **Số tham số Student:** {parameter_count:,}
- **Forward mean / median / p95:** {env['mean_forward_ms']} / {env['median_forward_ms']} / {env['p95_forward_ms']} ms/ảnh
- **Fusion mean (forward + chuyển màu):** {env['mean_fusion_ms']} ms/ảnh
- **Thời gian toàn vòng xử lý:** {env['loop_total_s']} giây
- **Peak GPU memory PyTorch:** {env['peak_gpu_allocated_mb']} MB

**Kết luận đúng phạm vi:** Đã chạy inference Student với trọng số pretrained và lưu 10 ảnh hợp nhất. Không suy ra độ chính xác segmentation, chất lượng vượt baseline, hay SOTA từ thử nghiệm này. Các metric EN, SD, SCD, MS-SSIM do thành viên phụ trách đánh giá thực hiện sau.

**Tài liệu bổ sung:** `environment.json`, `pip_freeze.txt`, `inference_times.csv`, `preview_ir_vis_fused.png`, `fused/`.
"""
(OUTPUT_DIR / 'README_results.md').write_text(report, encoding='utf-8')

# Unique ZIP basename per run, never delete an old archive.
zip_path = Path('/kaggle/working') / f'SAGE_LocBe_inference_{RUN_ID}.zip'
assert not zip_path.exists(), 'Output archive exists unexpectedly'
# Output contract: persist image-ID manifest and config for reproducibility.
import csv
config = {'run_id': RUN_ID, 'seed': SEED, 'method': 'SAGE pretrained Student',
          'checkpoint_sha256': checkpoint_sha, 'upstream_commit': repo_commit,
          'data_source': env['dataset_source'], 'split': 'FMB/test repack, not official test',
          'preprocessing': 'SAGE official dataset_test.Data; Y/IR input; RGB restored; 640x512 in this run if observed'}
(OUTPUT_DIR/'run_config.json').write_text(json.dumps(config, ensure_ascii=False,indent=2),encoding='utf-8')
with (OUTPUT_DIR/'manifest.csv').open('w',newline='',encoding='utf-8-sig') as f:
    writer=csv.DictWriter(f,fieldnames=['image_id','ir_path','vis_path','height','width','fused_path'])
    writer.writeheader()
    for rec in records:
        writer.writerow({'image_id':rec['stem'],'ir_path':rec['ir_path'],'vis_path':rec['vis_path'],
                         'height':rec['height'],'width':rec['width'],'fused_path':f"fused/{rec['stem']}.png"})
(OUTPUT_DIR/'logs').mkdir(exist_ok=False)
(OUTPUT_DIR/'logs/run.log').write_text(report,encoding='utf-8')
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=OUTPUT_DIR)
print(report)
print('ZIP kết quả:', zip_path)
print('Files:', [p.name for p in OUTPUT_DIR.iterdir()])
print('HOÀN TẤT — Chỉ đánh dấu hoàn thành sau khi các cell Kaggle chạy thành công!')

## Bàn giao

Sau khi **Run All** thành công, tại bảng **Output** của Kaggle tải ZIP `SAGE_LocBe_inference_<RUN_ID>.zip`; đồng thời tải notebook `.ipynb` đã chạy có lưu output cell và chia sẻ cho nhóm trưởng.

**Checklist trước khi gửi:**
- [ ] `THÀNH CÔNG: 10 fused PNG` ở Cell 5.
- [ ] `strict=True` load checkpoint thành công ở Cell 4.
- [ ] Preview IR / Visible / Fused nhìn hợp lý ở Cell 6.
- [ ] Có ZIP ảnh + file `inference_times.csv`, `environment.json`, `pip_freeze.txt`.
- [ ] Báo rõ nguồn dữ liệu dùng là FMB hay chỉ là ảnh test mẫu từ repo.

**Các lỗi hay gặp:** Thiếu GPU → đổi Accelerator; không clone được GitHub → bật Internet; thiếu 10 cặp ảnh → Add Input FMB; sai `DATA_ROOT` → trỏ vào thư mục **cha** chứa hai thư mục `Ir` và `Vis` (hoặc `IR` và `Visible`), không trỏ vào từng thư mục ảnh.
**Dữ liệu tự tải ở Cell 2B:** Ghi là `FMB (LC-State/RGB2IR_Translation repackaged test)` và không dùng số liệu đó để khẳng định tái lập đúng split benchmark chính thức.
